# 05 — Model Comparison & Winner Selection

**Vietnamese Punctuation Restoration · Phase 2, notebook 1 / 3**

> ### Notebook này KHÔNG train model và KHÔNG đọc `test.jsonl`.
> Nó chỉ đọc các artifact **validation** mà bốn notebook training đã sinh ra,
> so sánh chúng, và chọn ra model thắng cuộc.
>
> Sau khi cell ghi `model_selection.json` chạy xong, **winner bị khoá**. Không
> notebook nào sau đây được phép đổi winner — kể cả khi official test cho kết
> quả khác.

## 1. Vì sao chọn winner bằng validation chứ không bằng test?

Nếu chọn model bằng chính tập test rồi lại báo cáo điểm test của model đó, con
số báo cáo sẽ **lạc quan một cách có hệ thống**: ta đã dùng test để ra quyết
định, nên nó không còn là dữ liệu chưa từng thấy nữa. Đây là lỗi phương pháp
luận phổ biến nhất trong các đồ án machine learning.

Quy trình đúng, và cũng là quy trình dự án này tuân theo:

```
train.jsonl       → học tham số
validation.jsonl  → chọn epoch tốt nhất VÀ chọn model tốt nhất
test.jsonl        → đo một lần duy nhất, sau khi mọi quyết định đã chốt
```

## 2. Tiêu chí chọn

**Metric chính — validation Punctuation Macro-F1**

$$\text{PUNCT-F1} = \frac{F1_{COMMA} + F1_{PERIOD} + F1_{QUESTION}}{3}$$

Không dùng accuracy: nhãn `O` chiếm 91.1% nên một model đoán `O` cho mọi từ đã
đạt accuracy 0.911 mà không khôi phục được dấu câu nào.

**Tie-break — validation unweighted loss (thấp hơn thắng)**

Đây là chi tiết cần cẩn thận. Mỗi experiment ghi lại validation loss theo **hàm
loss của riêng nó**: E2 không trọng số, E3 trọng số inverse, E4 trọng số
sqrt-inverse. Ba con số đó nằm trên ba thang đo khác nhau, **không so sánh
được với nhau**. Vì vậy `scripts/compute_unweighted_validation_loss.py` đã nạp
lại cả bốn checkpoint và tính lại loss bằng **một hàm CrossEntropyLoss không
trọng số dùng chung**. Notebook này đọc con số đó.

Thứ tự sắp xếp cuối cùng là `(-PUNCT_F1, unweighted_loss, experiment_id)` nên
kết quả hoàn toàn tất định.

In [1]:
import os, sys, json, time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").exists(), f"Cannot locate the repo root from {Path.cwd()}"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from src.data.constants import LABELS, PUNCTUATION_LABELS, EXPERIMENT_IDS, OUTPUTS_DIR
from src.utils.io import read_json, write_json, write_csv
from src.utils.logging_utils import configure_stdout_utf8

configure_stdout_utf8()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

EVALUATION_DIR = OUTPUTS_DIR / "evaluation"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: .


## 3. Kiểm tra lại bốn lần training trước khi so sánh

So sánh bốn experiment chỉ có ý nghĩa nếu chúng thực sự chạy cùng một giao
thức. `scripts/verify_experiments.py` đã kiểm tra và ghi kết quả vào
`outputs/evaluation/training_verification.json`:

* đủ artifact bắt buộc, `status = COMPLETED`, checkpoint nạp được;
* `best_epoch` trong summary khớp với dòng tốt nhất trong `training_history.csv`;
* `punctuation_macro_f1` **được tính lại** từ confusion matrix lưu trên đĩa và
  khớp với giá trị đã lưu;
* cả bốn dùng **cùng hash train / validation / test**, cùng label mapping,
  cùng seed;
* không experiment nào ghi nhận đã dùng test split.

Ngoài ra `scripts/compute_unweighted_validation_loss.py` đã nạp lại từng
checkpoint và xác nhận nó **tái tạo đúng điểm validation đã báo cáo** — bằng
chứng rằng checkpoint trên đĩa đúng là checkpoint của epoch tốt nhất.

In [2]:
verification = read_json(EVALUATION_DIR / "training_verification.json")
unweighted   = read_json(EVALUATION_DIR / "validation_unweighted_loss.json")

assert verification["passed"], (
    "Training verification FAILED — không được so sánh model khi artifact chưa sạch:\n"
    + "\n".join(verification["errors"])
)
assert unweighted["all_checkpoints_reproduce_reported_scores"], (
    "Có checkpoint không tái tạo được điểm validation đã báo cáo."
)
assert unweighted["test_split_used"] is False

print("Training verification :", "PASSED" if verification["passed"] else "FAILED",
      f"({verification['num_warnings']} warning)")
print("Checkpoint reproduce  :", unweighted["all_checkpoints_reproduce_reported_scores"])
print()
cross = verification["cross_experiment"]
for key in ("train_hash", "validation_hash", "test_hash", "seed", "num_evaluated_tokens", "label2id"):
    rec = cross[key]
    sample = next(iter(rec["values"].values()))
    print(f"  [{'OK  ' if rec['consistent'] else 'FAIL'}] {key:<22} {str(sample)[:44]}")

Training verification : PASSED (0 warning)
Checkpoint reproduce  : True

  [OK  ] train_hash             138983bc2761546f13c9520d5339b2ea10e588b75605
  [OK  ] validation_hash        874133954b31fbfd73ccea2f84c69b15d106d4e8bc94
  [OK  ] test_hash              960cf879be7ac275ce16da846c9d37fd70838ada2c5b
  [OK  ] seed                   42
  [OK  ] num_evaluated_tokens   1977406
  [OK  ] label2id               {'COMMA': 1, 'O': 0, 'PERIOD': 2, 'QUESTION'


## 4. Bảng so sánh validation

Bảng dưới gom toàn bộ số liệu validation của bốn experiment. Mọi con số đọc
trực tiếp từ artifact, không có gì được nhập tay.

Cột `validation_loss_own_weighting` được giữ lại **chỉ để minh bạch** — nó cho
thấy vì sao không thể dùng cột đó làm tie-break: ba giá trị PhoBERT nằm trên ba
thang đo khác nhau.

In [3]:
from src.evaluation.selection import load_validation_candidates, select_winner

candidates = load_validation_candidates(EXPERIMENT_IDS)

comparison = pd.DataFrame([c.to_row() for c in candidates])
comparison = comparison.sort_values(
    ["validation_punctuation_macro_f1", "validation_loss_unweighted"],
    ascending=[False, True],
).reset_index(drop=True)
comparison.insert(0, "rank", range(1, len(comparison) + 1))

display(comparison)

,rank,experiment_id,model,weight_mode,best_epoch,validation_loss_unweighted,validation_accuracy,validation_macro_f1,validation_punctuation_macro_f1,f1_o,f1_comma,f1_period,f1_question,validation_loss_own_weighting
0,1,E2,vinai/phobert-base-v2,none,5,0.090323,0.966725,0.830614,0.778718,0.986302,0.726973,0.810835,0.798345,0.090390
1,2,E4,vinai/phobert-base-v2,sqrt_inverse,5,0.114305,0.957236,0.804645,0.745676,0.981549,0.700309,0.791063,0.745657,0.204728
2,3,E3,vinai/phobert-base-v2,inverse,5,0.185980,0.935833,0.763878,0.695324,0.969542,0.612241,0.758988,0.714742,0.302224
3,4,E1,"BiLSTM(emb=128, hidden=128x2)",none,11,0.158479,0.944757,0.714699,0.628241,0.974073,0.496436,0.686458,0.701829,0.158479


In [4]:

csv_path = EVALUATION_DIR / "validation_model_comparison.csv"
write_csv(csv_path, list(comparison.columns), comparison.values.tolist())

json_path = write_json(EVALUATION_DIR / "validation_model_comparison.json", {
    "split": "validation",
    "selection_metric": "punctuation_macro_f1",
    "tie_breaker": "unweighted_validation_loss",
    "test_split_used": False,
    "note": ("validation_loss_unweighted is computed for every model with ONE shared "
             "unweighted CrossEntropyLoss so the values are comparable. "
             "validation_loss_own_weighting is each experiment's own (weighted) loss and "
             "is NOT comparable across experiments."),
    "rows": comparison.to_dict(orient="records"),
})
print("Written:", csv_path.relative_to(PROJECT_ROOT))
print("Written:", json_path.relative_to(PROJECT_ROOT))

Written: outputs\evaluation\validation_model_comparison.csv
Written: outputs\evaluation\validation_model_comparison.json


## 5. Biểu đồ so sánh

Hai hình:

* **`validation_model_comparison.png`** — Punctuation Macro-F1 của bốn model
  cạnh nhau, cùng đường tham chiếu accuracy để thấy hai chỉ số kể hai câu
  chuyện rất khác nhau;
* **`validation_per_class_f1_comparison.png`** — F1 của `COMMA`, `PERIOD`,
  `QUESTION` theo từng model. Đây là hình cho thấy class weight thực sự tác
  động vào đâu.

In [5]:

order = comparison["experiment_id"].tolist()
punct = comparison["validation_punctuation_macro_f1"].tolist()
acc   = comparison["validation_accuracy"].tolist()
macro = comparison["validation_macro_f1"].tolist()

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(order))
bars = ax.bar(x, punct, width=0.55, color="#2a7fb8", label="Punctuation Macro-F1 (metric chọn winner)")
ax.plot(x, acc, "o--", color="#999999", label="Accuracy (KHÔNG dùng để chọn)")
ax.plot(x, macro, "s--", color="#c8663a", label="Macro-F1 4 lớp")

for xi, v in zip(x, punct):
    ax.text(xi, v + 0.012, f"{v:.4f}", ha="center", fontsize=10, fontweight="bold")

best_i = int(np.argmax(punct))
bars[best_i].set_color("#1a7a3c")
ax.set_xticks(x)
ax.set_xticklabels([f"{e}\n{w}" for e, w in
                    zip(order, comparison["weight_mode"])], fontsize=10)
ax.set_ylabel("score")
ax.set_ylim(0, 1.05)
ax.set_title("Validation comparison — winner được chọn bằng Punctuation Macro-F1")
ax.grid(axis="y", alpha=0.3)
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "validation_model_comparison.png", dpi=150)
plt.show()
print("Saved:", (FIGURES_DIR / "validation_model_comparison.png").relative_to(PROJECT_ROOT))

Saved: outputs\figures\validation_model_comparison.png


<USER_HOME>/AppData\Local\Temp\ipykernel_27432\220944158.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:

fig, ax = plt.subplots(figsize=(10, 5))
width = 0.2
x = np.arange(len(PUNCTUATION_LABELS))
colors = ["#7f7f7f", "#2a7fb8", "#c8663a", "#1a7a3c"]

for i, exp in enumerate(order):
    row = comparison[comparison["experiment_id"] == exp].iloc[0]
    vals = [row[f"f1_{c.lower()}"] for c in PUNCTUATION_LABELS]
    ax.bar(x + (i - 1.5) * width, vals, width, label=f"{exp} ({row['weight_mode']})",
           color=colors[i % len(colors)])
    for xi, v in zip(x + (i - 1.5) * width, vals):
        ax.text(xi, v + 0.008, f"{v:.3f}", ha="center", fontsize=7.5, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(PUNCTUATION_LABELS)
ax.set_ylabel("F1 (validation)")
ax.set_ylim(0, 1.0)
ax.set_title("F1 từng lớp dấu câu trên validation — nơi class weight thể hiện tác dụng")
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "validation_per_class_f1_comparison.png", dpi=150)
plt.show()
print("Saved:", (FIGURES_DIR / "validation_per_class_f1_comparison.png").relative_to(PROJECT_ROOT))

Saved: outputs\figures\validation_per_class_f1_comparison.png


<USER_HOME>/AppData\Local\Temp\ipykernel_27432\2273871999.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Chọn winner và KHOÁ

`select_winner()` chỉ nhận các đối tượng `ValidationCandidate` — một dataclass
đông cứng (frozen) mà **các trường của nó toàn bộ là số liệu validation**.
Không có chỗ nào để nhét metric test vào, nên việc rò rỉ test vào bước chọn
model là bất khả thi về mặt cấu trúc, chứ không chỉ là "chúng tôi hứa không
làm thế".

Sau cell dưới, `outputs/evaluation/model_selection.json` tồn tại với
`winner_locked = true`. Từ giờ mọi thành phần phía sau (notebook 06, 07,
inference, UI) đều đọc winner từ file này.

In [7]:
from src.evaluation.selection import write_model_selection

result = select_winner(candidates)

print(f"WINNER: {result.winner.experiment_id}  ({result.winner.model}, "
      f"weight_mode={result.winner.weight_mode})")
print(f"  validation Punctuation Macro-F1 : {result.winner.punctuation_macro_f1:.6f}")
print(f"  validation unweighted loss      : {result.winner.unweighted_validation_loss:.6f}")
print(f"  best epoch                      : {result.winner.best_epoch}")
print(f"  checkpoint                      : {result.winner.checkpoint_dir}")
print()
print(f"  runner-up      : {result.runner_up.experiment_id if result.runner_up else '-'}")
print(f"  margin         : {result.margin_over_runner_up:.6f}"
      if result.margin_over_runner_up is not None else "  margin: n/a")
print(f"  tie-break used : {result.tie_breaker_used}")

selection_path = write_model_selection(result)
print("\nWritten:", selection_path.relative_to(PROJECT_ROOT))
print(json.dumps({k: v for k, v in read_json(selection_path).items()
                  if k in ("selection_split", "selection_metric", "tie_breaker",
                           "winner", "test_was_used_for_selection", "winner_locked")},
                 indent=2, ensure_ascii=False))

2026-08-10 05:59:40 | INFO    | src.evaluation.selection | Winner: E2 (validation punctuation_macro_f1 = 0.778718, unweighted loss = 0.090323, tie_breaker_used=False)


WINNER: E2  (vinai/phobert-base-v2, weight_mode=none)
  validation Punctuation Macro-F1 : 0.778718
  validation unweighted loss      : 0.090323
  best epoch                      : 5
  checkpoint                      : outputs/checkpoints/E2

  runner-up      : E4
  margin         : 0.033041
  tie-break used : False
2026-08-10 05:59:40 | INFO    | src.evaluation.selection | Winner E2 locked in outputs\evaluation\model_selection.json



Written: outputs\evaluation\model_selection.json
{
  "selection_split": "validation",
  "selection_metric": "punctuation_macro_f1",
  "tie_breaker": "unweighted_validation_loss",
  "winner": "E2",
  "test_was_used_for_selection": false,
  "winner_locked": true
}


## 7. Đọc kết quả

Bảng và biểu đồ ở trên trả lời ba câu hỏi nghiên cứu của đồ án:

1. **Tiền huấn luyện tiếng Việt có đáng không?** So E1 (BiLSTM from scratch)
   với các model PhoBERT.
2. **Class weight có giúp không?** So E2 (không trọng số) với E3 (inverse) và
   E4 (sqrt-inverse). Chú ý cả precision lẫn recall của lớp hiếm, đừng chỉ nhìn
   recall.
3. **Chênh lệch có đủ lớn để kết luận không?** Nếu hai model chênh nhau vài
   phần nghìn thì kết luận trung thực là "không khác biệt đáng kể", chứ không
   phải cố tuyên bố một người thắng.

Cell dưới in ra các so sánh đó bằng số liệu thực tế.

In [8]:
by_id = {c.experiment_id: c for c in candidates}

print("So sánh theo cặp (validation Punctuation Macro-F1)\n")
def delta(a, b, label):
    d = by_id[a].punctuation_macro_f1 - by_id[b].punctuation_macro_f1
    rel = 100 * d / by_id[b].punctuation_macro_f1
    print(f"  {label:<46} {by_id[a].punctuation_macro_f1:.4f} vs {by_id[b].punctuation_macro_f1:.4f}"
          f"   Δ = {d:+.4f} ({rel:+.1f}%)")

delta("E2", "E1", "PhoBERT (E2) so với BiLSTM (E1)")
delta("E3", "E2", "inverse weight (E3) so với no weight (E2)")
delta("E4", "E2", "sqrt-inverse weight (E4) so với no weight (E2)")
delta("E4", "E3", "sqrt-inverse (E4) so với inverse (E3)")

print("\nRecall / precision của lớp hiếm QUESTION\n")
print(f"  {'exp':<5}{'weight':<14}{'precision':>11}{'recall':>10}{'F1':>10}")
for e in EXPERIMENT_IDS:
    c = by_id[e]
    print(f"  {e:<5}{c.weight_mode:<14}{c.precision_per_class['QUESTION']:>11.4f}"
          f"{c.recall_per_class['QUESTION']:>10.4f}{c.f1_per_class['QUESTION']:>10.4f}")

print("\nRecall / precision của COMMA (lớp dấu câu phổ biến nhất)\n")
print(f"  {'exp':<5}{'weight':<14}{'precision':>11}{'recall':>10}{'F1':>10}")
for e in EXPERIMENT_IDS:
    c = by_id[e]
    print(f"  {e:<5}{c.weight_mode:<14}{c.precision_per_class['COMMA']:>11.4f}"
          f"{c.recall_per_class['COMMA']:>10.4f}{c.f1_per_class['COMMA']:>10.4f}")

So sánh theo cặp (validation Punctuation Macro-F1)

  PhoBERT (E2) so với BiLSTM (E1)                0.7787 vs 0.6282   Δ = +0.1505 (+24.0%)
  inverse weight (E3) so với no weight (E2)      0.6953 vs 0.7787   Δ = -0.0834 (-10.7%)
  sqrt-inverse weight (E4) so với no weight (E2) 0.7457 vs 0.7787   Δ = -0.0330 (-4.2%)
  sqrt-inverse (E4) so với inverse (E3)          0.7457 vs 0.6953   Δ = +0.0504 (+7.2%)

Recall / precision của lớp hiếm QUESTION

  exp  weight          precision    recall        F1
  E1   none               0.7086    0.6952    0.7018
  E2   none               0.7513    0.8516    0.7983
  E3   inverse            0.5739    0.9472    0.7147
  E4   sqrt_inverse       0.6198    0.9357    0.7457

Recall / precision của COMMA (lớp dấu câu phổ biến nhất)

  exp  weight          precision    recall        F1
  E1   none               0.6599    0.3979    0.4964
  E2   none               0.7437    0.7110    0.7270
  E3   inverse            0.4820    0.8389    0.6122
  E4   sqrt_inv

## 8. Winner đã bị khoá — bước tiếp theo

`outputs/evaluation/model_selection.json` giờ có `winner_locked = true` và
`test_was_used_for_selection = false`.

**Cam kết:** winner này sẽ không thay đổi. Nếu notebook 07 (post-hoc) cho thấy
một model khác đạt điểm test cao hơn, điều đó được **ghi nhận và thảo luận**,
nhưng winner vẫn giữ nguyên. Đổi winner sau khi nhìn test chính là hình thức
rò rỉ test mà toàn bộ quy trình này được thiết kế để ngăn chặn.

Tiếp theo:

* `notebooks/06_Final_Test_Evaluation.ipynb` — chạy official test **một lần
  duy nhất** trên winner;
* `notebooks/07_Posthoc_Test_Comparison.ipynb` — (tuỳ chọn) so sánh mô tả tất
  cả model + baseline trên test.

In [9]:

expected = [
    EVALUATION_DIR / "validation_model_comparison.csv",
    EVALUATION_DIR / "validation_model_comparison.json",
    EVALUATION_DIR / "model_selection.json",
    FIGURES_DIR / "validation_model_comparison.png",
    FIGURES_DIR / "validation_per_class_f1_comparison.png",
]
ok = True
for p in expected:
    ok &= p.exists()
    print(f"  [{'OK  ' if p.exists() else 'MISS'}] {p.relative_to(PROJECT_ROOT)}")

sel = read_json(EVALUATION_DIR / "model_selection.json")
checks = {
    "winner_locked is true": sel["winner_locked"] is True,
    "test_was_used_for_selection is false": sel["test_was_used_for_selection"] is False,
    "selection_split is validation": sel["selection_split"] == "validation",
    "selection_metric is punctuation_macro_f1": sel["selection_metric"] == "punctuation_macro_f1",
}
print()
for k, v in checks.items():
    ok &= v
    print(f"  [{'OK  ' if v else 'FAIL'}] {k}")

print("\n" + "=" * 78)
print(f"NOTEBOOK 05 {'COMPLETE' if ok else 'INCOMPLETE'} — winner = {sel['winner']} (locked)")
print("test.jsonl was not read in this notebook.")
print("=" * 78)

  [OK  ] outputs\evaluation\validation_model_comparison.csv
  [OK  ] outputs\evaluation\validation_model_comparison.json
  [OK  ] outputs\evaluation\model_selection.json
  [OK  ] outputs\figures\validation_model_comparison.png
  [OK  ] outputs\figures\validation_per_class_f1_comparison.png

  [OK  ] winner_locked is true
  [OK  ] test_was_used_for_selection is false
  [OK  ] selection_split is validation
  [OK  ] selection_metric is punctuation_macro_f1

NOTEBOOK 05 COMPLETE — winner = E2 (locked)
test.jsonl was not read in this notebook.
